# 1. Generate Sales Forecast Consolidation Data

This workbook runs the synthetic sales forecast data generation process. It populates regional PostgreSQL databases for US and EU, and creates a local CSV spreadsheet for the UK sales team. The dataset spans 3 years of historical opportunities (mid-2023 to mid-2026), modeling typical sales cycles and compound annual growth.

In [ ]:
%load_ext autoreload
%autoreload 2
from sales_forecast_consolidation_setup.generate_synthetic_sales_data import generate_data
generate_data()

## Load and Analyze the Data

Let's load the generated datasets into Pandas dataframes to examine their volume, distribution, and seasonal trends.

In [ ]:
import pandas as pd
import psycopg2
import warnings

# Suppress Pandas UserWarning regarding raw DBAPI2 connections
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

# 1. Load UK CSV Spreadsheet
uk_df = pd.read_csv('uk_sales_forecast.csv')
print(f"UK Forecast Opportunities (Spreadsheet): {len(uk_df)}")

# 2. Load US and EU Sales from PostgreSQL
conn = psycopg2.connect(
    host='host.docker.internal',
    port=5442,
    dbname='coco_pharma',
    user='postgres',
    password='egeria'
)

us_df = pd.read_sql_query('SELECT * FROM us_sales.us_sales_forecast', conn)
eu_df = pd.read_sql_query('SELECT * FROM eu_sales.eu_sales_forecast', conn)
conn.close()

print(f"US Forecast Opportunities (PostgreSQL): {len(us_df)}")
print(f"EU Forecast Opportunities (PostgreSQL): {len(eu_df)}")

## 3-Year Opportunity Summary and YoY Growth

In [ ]:
# Add a Year-Month column for trend analysis
# Note: PostgreSQL folds unquoted column names to lowercase, so 'Date' is retrieved as 'date'
us_df['date_parsed'] = pd.to_datetime(us_df['date'])
us_df['year'] = us_df['date_parsed'].dt.year
us_yearly = us_df.groupby('year')['forecastamount'].agg(['count', 'sum']).rename(columns={'count': 'deal_count', 'sum': 'total_amount_usd'})

print("US Yearly Sales Pipeline Growth:")
print(us_yearly)